# Varve age-depth model figure

Run the cells from top to bottom (**Runtime → Run all** works too).

1. **Step 1** installs the Excel reader.
2. **Step 2** loads the plotting code. You don't need to change it.
3. **Step 3** asks you to upload your varve-counting Excel file.
4. **Step 4** is where you set the axis limits and options, then draws the figure.
5. **Step 5** downloads the figure as PNG and PDF.

**Excel layout:** one sheet per core (sheet name = core ID) with a `Depth (cm)` column and a `Year (CE)` column.
Other layouts that also work are explained at the top of Step 2.

In [ ]:
# Step 1: install the Excel reader (pandas, numpy and matplotlib are already in Colab)
!pip install -q openpyxl

In [ ]:
# Step 2: plotting code (no changes needed)
"""
Varve-counted age-depth model plot.

Reads varve counts for one or more cores from an Excel workbook and draws
all cores on one age-depth figure: age (yr CE) on a top x-axis, depth (cm)
increasing downward, one stepped line per core.

Accepted workbook layouts (detected automatically)
--------------------------------------------------
1. One sheet per core (sheet name = core ID), with a depth column and a
   year column, e.g.

       Depth (cm) | Year (CE)
       0.00       | 2023
       0.42       | 2022
       ...

2. One sheet holding several cores side by side, as depth/year column
   pairs whose headers start with the core ID, e.g.

       GUAC-22A-1G-1 Depth | GUAC-22A-1G-1 Year | GUAC-23A-1G-1 Depth | ...

3. Varve thickness only (no year column): a column of varve thicknesses
   in mm, one row per varve from the top down. Depth is the cumulative
   thickness and the year is counted back from --top-year.

Column headers are matched case-insensitively: "depth"; "year", "age",
"ce" or "ad"; "thick" (thickness in mm). Rows with blanks are skipped.

Usage
-----
    python plot_age_depth.py varve_counts.xlsx
    python plot_age_depth.py varve_counts.xlsx -o age_depth.png \
        --xlim 1700 2050 --ylim 90 0 --style steps
"""

import argparse
import re
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, LogLocator
import numpy as np
import pandas as pd

# Colours matched to the Excel chart (Office theme, in series order).
PLAIN = FuncFormatter(lambda v, _: f"{v:g}")  # 0.1, 1, 10 instead of 10^-1

CORE_COLORS = [
    "#156082",  # dark teal-blue
    "#E97132",  # orange
    "#196B24",  # dark green
    "#0F9ED5",  # light blue
    "#A02B93",  # purple
    "#4EA72E",  # light green
    "#0E2841",  # navy
    "#8B3A0F",  # brown
    "#5E5E5E",  # grey (extra cores)
    "#C9A100",  # mustard
]

DEPTH_RE = re.compile(r"depth", re.I)
YEAR_RE = re.compile(r"\b(year|age|ce|ad)\b|year|age", re.I)
THICK_RE = re.compile(r"thick", re.I)


def _find_cols(columns, pattern):
    return [c for c in columns if pattern.search(str(c))]


def _core_name(header, fallback):
    """Strip the depth/year words and units from a header to get the core ID."""
    name = re.sub(r"\(.*?\)|depth|year|age|thickness|\bce\b|\bad\b|\bcm\b|\bmm\b",
                  "", str(header), flags=re.I)
    name = name.strip(" _-:")
    return name or fallback


def _clean(depth, year):
    df = pd.DataFrame({"depth": pd.to_numeric(depth, errors="coerce"),
                       "year": pd.to_numeric(year, errors="coerce")}).dropna()
    return df.sort_values(["depth", "year"], ascending=[True, False]).reset_index(drop=True)


def read_cores(xlsx_path, top_year=None):
    """Return {core_id: DataFrame(depth, year)} from every sheet of the workbook."""
    sheets = pd.read_excel(xlsx_path, sheet_name=None)
    cores = {}
    for sheet_name, df in sheets.items():
        df = df.dropna(how="all").dropna(axis=1, how="all")
        if df.empty:
            continue
        depth_cols = _find_cols(df.columns, DEPTH_RE)
        year_cols = [c for c in _find_cols(df.columns, YEAR_RE) if c not in depth_cols]
        thick_cols = _find_cols(df.columns, THICK_RE)

        if depth_cols and year_cols:
            if len(depth_cols) != len(year_cols):
                raise ValueError(f"Sheet '{sheet_name}': {len(depth_cols)} depth columns "
                                 f"but {len(year_cols)} year columns.")
            for d, y in zip(depth_cols, year_cols):
                name = sheet_name if len(depth_cols) == 1 else _core_name(d, sheet_name)
                cores[name] = _clean(df[d], df[y])
        elif thick_cols:
            if top_year is None:
                raise ValueError(f"Sheet '{sheet_name}' has only varve thickness; "
                                 "pass --top-year (year of the topmost varve).")
            for t in thick_cols:
                name = sheet_name if len(thick_cols) == 1 else _core_name(t, sheet_name)
                thick_cm = pd.to_numeric(df[t], errors="coerce").dropna().to_numpy() / 10.0
                depth = np.concatenate([[0.0], np.cumsum(thick_cm)])
                year = top_year - np.arange(len(depth))
                cores[name] = _clean(depth, year)
        else:
            print(f"Skipping sheet '{sheet_name}': no depth/year or thickness columns found.")
    if not cores:
        raise ValueError("No core data found in the workbook.")
    return cores


def _nice_limits(lo, hi, step):
    return np.floor(lo / step) * step, np.ceil(hi / step) * step


def plot_age_depth(cores, out_path, xlim=None, ylim=None, style="steps",
                   title=None, xstep=50, ystep=10):
    fig, ax = plt.subplots(figsize=(11, 8))

    for i, (name, df) in enumerate(cores.items()):
        ax.plot(df["year"], df["depth"],
                color=CORE_COLORS[i % len(CORE_COLORS)], lw=1.6, label=name,
                drawstyle="steps-post" if style == "steps" else "default")

    all_years = pd.concat([d["year"] for d in cores.values()])
    all_depths = pd.concat([d["depth"] for d in cores.values()])
    if xlim is None:
        xlim = _nice_limits(all_years.min(), all_years.max(), xstep)
    if ylim is None:
        ylim = (_nice_limits(0, all_depths.max(), ystep)[1], 0)
    ax.set_xlim(xlim)
    ax.set_ylim(max(ylim), min(ylim))  # depth increases downward

    # Age axis on top, like the Excel chart.
    ax.xaxis.tick_top()
    ax.xaxis.set_label_position("top")
    ax.set_xticks(np.arange(min(xlim), max(xlim) + xstep, xstep))
    ax.set_yticks(np.arange(min(ylim), max(ylim) + ystep, ystep))
    ax.set_xlabel("Age (yr CE)", fontsize=12, labelpad=8)
    ax.set_ylabel("Depth (cm)", fontsize=12)
    ax.tick_params(labelsize=10, direction="out")
    for side in ("right", "bottom"):
        ax.spines[side].set_visible(False)

    ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.6), frameon=False,
              fontsize=11, handlelength=1.5)
    if title:
        ax.set_title(title, fontsize=13, pad=40)

    fig.tight_layout()
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    print(f"Saved {out_path}")
    return fig, ax


def varve_thickness(df):
    """Per-year varve thickness (mm) from the depth/year series of one core.

    Rows sharing a year (e.g. top and base of an event layer) are merged, and
    each year's thickness is the depth to the top of the next-older varve.
    The oldest varve has no base, so it is dropped.
    """
    tops = df.groupby("year")["depth"].min().sort_index(ascending=False)
    thick_mm = (tops.shift(-1) - tops) * 10.0
    out = pd.DataFrame({"year": tops.index, "depth": tops.values,
                        "thickness": thick_mm.values}).dropna()
    return out[out["thickness"] > 0].reset_index(drop=True)


def plot_age_depth_panel(cores, out_path, xlim=None, ylim=None, style="steps",
                         title=None, xstep=50, ystep=10, event_mm=None):
    """Three linked scales in one figure.

    (a) age-depth curves; (b) varve thickness vs depth, sharing (a)'s depth
    axis; (c) varve thickness vs age, one strip per core, sharing (a)'s age axis.
    """
    names = list(cores)
    colors = {n: CORE_COLORS[i % len(CORE_COLORS)] for i, n in enumerate(names)}
    thick = {n: varve_thickness(df) for n, df in cores.items()}
    drawstyle = "steps-post" if style == "steps" else "default"

    all_years = pd.concat([d["year"] for d in cores.values()])
    all_depths = pd.concat([d["depth"] for d in cores.values()])
    all_thick = pd.concat([t["thickness"] for t in thick.values()])
    if xlim is None:
        xlim = _nice_limits(all_years.min(), all_years.max(), xstep)
    if ylim is None:
        ylim = (_nice_limits(0, all_depths.max(), ystep)[1], 0)
    tlim = (10 ** np.floor(np.log10(all_thick.min())),
            10 ** np.ceil(np.log10(all_thick.max())))

    n = len(names)
    fig = plt.figure(figsize=(12, 8.5 + 0.45 * n))
    outer = fig.add_gridspec(2, 2, width_ratios=[3.2, 1], height_ratios=[3, 0.16 * n + 0.4],
                             wspace=0.1, hspace=0.08)
    ax = fig.add_subplot(outer[0, 0])
    axb = fig.add_subplot(outer[0, 1], sharey=ax)
    strips = outer[1, 0].subgridspec(n, 1, hspace=0)
    axc = [fig.add_subplot(strips[i], sharex=ax) for i in range(n)]

    # (a) age-depth model
    for name, df in cores.items():
        ax.plot(df["year"], df["depth"], color=colors[name], lw=1.5,
                drawstyle=drawstyle, label=name)
    ax.set_xlim(xlim)
    ax.set_ylim(max(ylim), min(ylim))
    ax.set_xticks(np.arange(min(xlim), max(xlim) + xstep, xstep))
    ax.set_yticks(np.arange(min(ylim), max(ylim) + ystep, ystep))
    ax.xaxis.tick_top()
    ax.xaxis.set_label_position("top")
    ax.set_xlabel("Age (yr CE)", fontsize=12, labelpad=8)
    ax.set_ylabel("Depth (cm)", fontsize=12)
    ax.grid(color="0.9", lw=0.6)
    ax.set_axisbelow(True)

    # (b) varve thickness vs depth
    for name, t in thick.items():
        axb.plot(t["thickness"], t["depth"], color=colors[name], lw=0.8, alpha=0.85,
                 drawstyle="steps-pre")
    axb.set_xscale("log")
    axb.set_xlim(tlim)
    axb.xaxis.set_major_locator(LogLocator(base=10, numticks=10))
    axb.xaxis.set_major_formatter(PLAIN)
    axb.xaxis.tick_top()
    axb.xaxis.set_label_position("top")
    axb.set_xlabel("Varve thickness (mm)", fontsize=12, labelpad=8)
    axb.tick_params(labelleft=False)
    axb.grid(color="0.9", lw=0.6, which="major")
    axb.set_axisbelow(True)

    # (c) varve thickness vs age, one strip per core
    for a, name in zip(axc, names):
        t = thick[name]
        a.fill_between(t["year"], tlim[0], t["thickness"], step="post",
                       color=colors[name], alpha=0.25, lw=0)
        a.plot(t["year"], t["thickness"], color=colors[name], lw=0.8, drawstyle="steps-post")
        a.set_yscale("log")
        a.set_ylim(tlim)
        a.set_yticks([v for v in (1, 10, 100) if tlim[0] < v < tlim[1]])
        a.yaxis.set_major_formatter(PLAIN)
        a.minorticks_off()
        a.tick_params(axis="y", labelsize=7, length=2, pad=1)
        a.text(1.005, 0.5, name, transform=a.transAxes, va="center", fontsize=9,
               color=colors[name])
        a.grid(axis="x", color="0.9", lw=0.6)
        a.set_axisbelow(True)
        if a is not axc[-1]:
            a.tick_params(labelbottom=False, bottom=False)
    axc[-1].set_xlabel("Age (yr CE)", fontsize=12)
    axc[n // 2].set_ylabel("Varve\nthickness\n(mm, log)", fontsize=10,
                           rotation=0, ha="right", va="center", labelpad=10)

    if event_mm:
        axb.axvline(event_mm, color="0.35", ls="--", lw=0.9)
        axb.text(event_mm, max(ylim), f" {event_mm:g} mm", fontsize=8, color="0.35",
                 va="bottom", rotation=90)
        for a in axc:
            a.axhline(event_mm, color="0.35", ls="--", lw=0.7)

    for a, label in ((ax, "a"), (axb, "b"), (axc[0], "c")):
        a.text(0.012, 0.985 if a is not axc[0] else 0.92, label, transform=a.transAxes,
               fontsize=13, fontweight="bold", va="top", zorder=5,
               bbox=dict(facecolor="white", edgecolor="none", pad=1.5))

    ax.legend(loc="lower right", frameon=True, framealpha=0.9, edgecolor="0.8",
              fontsize=9, ncol=2, title="Core", title_fontsize=9)
    if title:
        fig.suptitle(title, fontsize=14)

    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    print(f"Saved {out_path}")
    return fig


In [ ]:
# Step 3: upload your varve-counting Excel file
from google.colab import files

uploaded = files.upload()
EXCEL_FILE = next(iter(uploaded))
print("Using", EXCEL_FILE)

In [ ]:
# Step 4: settings, then draw the figure
XLIM = (1700, 2050)   # age axis (yr CE); set to None for automatic
YLIM = (90, 0)        # depth axis (cm), bottom then top; set to None for automatic
EVENT_MM = 10         # dashed varve-thickness line in mm; set to None for no line
TOP_YEAR = None       # only for thickness-only sheets: year of the topmost varve, e.g. 2023
LAYOUT = "panel"      # "panel" = three panels with thickness; "simple" = age-depth curves only
TITLE = None          # e.g. "Lake Lachuá varve chronology"

cores = read_cores(EXCEL_FILE, top_year=TOP_YEAR)
for name, df in cores.items():
    print(f"  {name}: {len(df)} points, {df['depth'].max():.1f} cm, "
          f"{df['year'].min():.0f}-{df['year'].max():.0f} CE")

if LAYOUT == "panel":
    plot_age_depth_panel(cores, "age_depth_model.png", xlim=XLIM, ylim=YLIM,
                         title=TITLE, event_mm=EVENT_MM)
    plot_age_depth_panel(cores, "age_depth_model.pdf", xlim=XLIM, ylim=YLIM,
                         title=TITLE, event_mm=EVENT_MM)
else:
    plot_age_depth(cores, "age_depth_model.png", xlim=XLIM, ylim=YLIM, title=TITLE)
    plot_age_depth(cores, "age_depth_model.pdf", xlim=XLIM, ylim=YLIM, title=TITLE)
plt.show()

In [ ]:
# Step 5: download the figure
files.download("age_depth_model.png")
files.download("age_depth_model.pdf")